In [40]:
import math, random

import numpy as np
import pandas as pd
import statsmodels.api as sm

from tqdm import tqdm
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

### Library

In [41]:
""" Data generators/parsers """

def generate_scenario_data(rows: int, scenario: str, miss_col: int):
    # Generate data as in https://www.nature.com/articles/s41467-020-19270-2 scenarios
    assert scenario in {"linear", "logistic"}, f"Invalid scenario for MI data {scenario}"
    feature_count = 2

    # x_2 is from U(-3, 3)
    x_2 = np.random.rand(rows) * 6 - 3

    if scenario == "linear":
        # Linear x_1 is from normal(mu=0.2 - 0.5*x_2, sigma=1.0)
        x_1 = np.random.normal(loc=0.2 - 0.5 * x_2, scale=1.0)
    else:
        # Logistic x_1 is from bernoulli(n=1, p=1 / (1 + exp(-0.2 + 0.5*x_2)))
        x_1 = np.random.binomial(n=1, p=1 / (1 + np.exp(-0.2 + 0.5 * x_2)))
    
    non_tampered_data = np.hstack((np.expand_dims(x_1, axis=1), np.expand_dims(x_2, axis=1)))
    
    miss_rows = []
    tampered_data = non_tampered_data.copy()
    
    non_tampered_labels = non_tampered_data @ np.ones((feature_count, 1)) + 1.0  # Weights and bias are ones
    non_tampered_labels += np.random.normal(loc=0.0, scale=1.0, size=non_tampered_labels.shape)  # Add some noise
    for i in range(rows):
        # The variable is missing with probability 1 / (1 + math.exp(-0.3 + 0.2 * labels[i][0] - 0.1 * x_2[i]))
        if (1 / (1 + math.exp(-0.3 + 0.2 * non_tampered_labels[i][0] - 0.1 * x_2[i]))) > random.random():
            tampered_data[i][miss_col] = np.nan
            miss_rows.append(i)

    print(f"Generated data missing rate: {len(miss_rows)}/{len(non_tampered_data)}")
    return tampered_data, non_tampered_labels, non_tampered_data, miss_rows

def parse_real_data(outlier_threshold):
    # Real data
    df = pd.read_csv("../../data/mi/real/gcasr.csv")
    binary_columns = ["RaceAA", "RaceW", "HlthInsM", "HlthInsN", "Day", "NPO", "MedHisST", "MedHisTI", "MedHisVP", "MedHisFHSTK"]
    continuous_columns = ["NIHStrkS", "EMSNote", "LipTotal", "Age", "Gender"]
    col_types = ["lin" if i < len(continuous_columns) else "log" for i in range(len(binary_columns) + len(continuous_columns))]
    df_of_interest = df[continuous_columns + binary_columns]

    # Scale binary columns from [0, 1] to [-1, 1]
    df_of_interest[binary_columns] = df_of_interest[binary_columns] * 2 - 1
    # Normalize continuous columns
    df_of_interest[continuous_columns] = (df_of_interest[continuous_columns] - df_of_interest[continuous_columns].mean()) / df_of_interest[continuous_columns].std()

    # Fill the missing data for numerical feasibility
    real_data = df_of_interest.to_numpy()
    # Drop the missing data to get the training data
    real_complete_data = df_of_interest.dropna()

    # Get labels
    x = df["x"].fillna(0)
    # Reduce the outliers
    x[x < -outlier_threshold] = -outlier_threshold
    x[x > outlier_threshold] = outlier_threshold
    # Normalize
    x = (x - x.mean()) / x.std()
    real_labels = x.to_numpy()

    # Get missing rows
    real_miss_rows = df_of_interest.isna().to_numpy().astype(int).T

    return real_data, real_complete_data.to_numpy(), real_labels, real_miss_rows, col_types

def export_data(name, data, labels, complete_data, miss_rows, miss_val):
    np.savetxt(f"../../data/mi/{name}/tampered_data.txt", np.nan_to_num(data, nan=miss_val), fmt='%10.2f')
    np.savetxt(f"../../data/mi/{name}/non_tampered_data.txt", complete_data, fmt='%10.2f')
    np.savetxt(f"../../data/mi/{name}/non_tampered_labels.txt", labels, fmt='%10.3f')
    np.savetxt(f"../../data/mi/{name}/miss_rows.txt", miss_rows, fmt='%d')

In [42]:
""" Imputers """

class Imputer:
    def __init__(self, model):
        self.model = model
    
    @staticmethod
    def split_train_test(data, target_col: int, miss_rows):
        m, n = data.shape
        miss_count = len(miss_rows)

        complete_rows = np.zeros((m - miss_count, n - 1))
        complete_labels = np.zeros((m - miss_count,))
        incomplete_rows = np.zeros((miss_count, n - 1))

        x_idx = 0
        x_com_idx = 0
        for i in range(m):
            row = data[i]
            row_complement = np.delete(row, target_col)
            if i in miss_rows:
                incomplete_rows[x_com_idx] = row_complement
                x_com_idx += 1
            else:
                complete_rows[x_idx] = row_complement
                complete_labels[x_idx] = row[target_col]
                x_idx += 1
        
        return complete_rows, incomplete_rows, complete_labels
    
    def fit(self, complete_data, complete_labels):
        self.model.fit(X=complete_data, y=complete_labels)
    
    def impute(self, data, miss_rows, miss_col: int, noise: bool):
        complete_data, incomplete_data, complete_labels = Imputer.split_train_test(data, miss_col, miss_rows)
        
        if len(incomplete_data) == 0:
            return complete_data.copy()
        
        self.model.fit(X=complete_data, y=complete_labels)
        imputed_y = self.model.predict(incomplete_data)
        if noise:
            imputed_y += np.random.normal(0.0, 0.5, size=imputed_y.shape)

        idx = 0
        imputed_data = data.copy()
        for i in miss_rows:
            imputed_data[i, miss_col] = imputed_y.ravel()[idx]
            idx += 1

        return imputed_data
    
    def impute(self, data, incomplete_data, miss_rows, miss_col, noise):
        imputed_y = self.model.predict(incomplete_data)
        if noise:
            imputed_y += np.random.normal(0.0, 0.5, size=imputed_y.shape)

        idx = 0
        imputed_data = data.copy()
        for i in miss_rows:
            imputed_data[i, miss_col] = imputed_y.ravel()[idx]
            idx += 1

        return imputed_data
    
    def impute_inplace(self, data, mask, miss_col, noise):
        if mask.sum() == 0:  # Nothing to impute --- no missing data
            return
        
        reference_data = np.hstack((data[:, :miss_col], data[:, miss_col+1:]))
        imputed_y = self.model.predict(reference_data[np.where(mask == 1)])
        if noise and isinstance(self.model, LinearRegression):
            imputed_y += np.random.normal(0.0, 0.5, size=imputed_y.shape)

        idx = 0
        for i in range(len(mask)):
            if mask[i]:
                data[i, miss_col] = imputed_y.ravel()[idx]
                idx += 1
    

class MI:
    def __init__(self, factor: int, impute_model, fit_model):
        self.imputer = Imputer(impute_model)
        self.model = fit_model
        self.factor = factor

    @staticmethod
    def rubin(weights):
        return np.mean(weights, axis=0)
    
    def fit(self, data, labels, miss_rows, miss_col, noise):
        complete_data, incomplete_data, complete_labels = Imputer.split_train_test(data, miss_col, miss_rows)
        
        imputed_data = []
        self.imputer.fit(complete_data=complete_data, complete_labels=complete_labels)
        
        imputed_data = [self.imputer.impute(
            data=data, incomplete_data=incomplete_data,
            miss_rows=miss_rows, miss_col=miss_col, noise=noise) for _ in range(self.factor)]
        
        weights = np.zeros((self.factor, data.shape[1]))
        for i, data in enumerate(imputed_data):
            self.model.fit(X=data, y=labels)
            weights[i] = self.model.coef_.copy()
        
        self.model.coef_ = MI.rubin(weights)
        return self, imputed_data
    

class BatchMI:
    def __init__(self, factor: int, impute_models, fit_model):
        self.imputers = [Imputer(impute_model) for impute_model in impute_models]
        self.model = fit_model
        self.factor = factor

    def fit(self, data, complete_data, labels, miss_rows):
        miss_col = 0
        for imputer in self.imputers:
            train_data = np.hstack((complete_data[:, :miss_col], complete_data[:, miss_col+1:]))
            train_labels = complete_data[:, miss_col:miss_col+1]
            
            imputer.fit(complete_data=train_data, complete_labels=train_labels.ravel())
            miss_col += 1
        
        imputed_data = []
        weights = np.zeros((self.factor, data.shape[1]))
        for i in range(self.factor):
            inter_imputed_data = data.copy()
            self.impute_inplace(inter_imputed_data, miss_rows)
            imputed_data.append(inter_imputed_data)
            self.model.fit(X=inter_imputed_data, y=labels)
            weights[i] = self.model.coef_.copy()
        
        self.model.coef_ = MI.rubin(weights)
        return self, imputed_data
    
    def impute_inplace(self, incomplete_data, miss_rows):
        miss_col = 0
        for imputer in self.imputers:
            imputer.impute_inplace(
                data=incomplete_data,
                mask=miss_rows[miss_col], miss_col=miss_col, noise=True)
            miss_col += 1

In [43]:
""" PyMICE """

def pymice(incomplete_data):
    imp_mean = IterativeImputer()
    imp_mean.fit(incomplete_data)
    return imp_mean.transform(incomplete_data)

In [44]:
""" Evaluators """

def sign(x):
    return -1 if x < 0 else 1 if x > 0 else 0

def get_stat_sig(X, y):
    X2 = sm.add_constant(X)
    est = sm.OLS(np.expand_dims(y, axis=1), X2)
    est2 = est.fit()
    return est2.params, est2.pvalues

def discrepancies(data_instance, mdni_stats, sequre_stats, verbose=False):
    mdni_coeff, mdni_pval = mdni_stats
    sequre_coeff, sequre_pval = sequre_stats

    discr = 0
    for i in range(len(mdni_coeff)):
        if mdni_pval[i] <= 0.05 and (sequre_pval[i] > 0.05 or sign(mdni_coeff[i]) != sign(sequre_coeff[i])):
            if verbose:
                print(f"Discrepancy found at var {i} and instance {data_instance}! SIMICE p_val: {mdni_pval[i]}, Sequre p_val: {sequre_pval[i]}, SIMICE coeff: {mdni_coeff[i]}, Sequre coeff: {sequre_coeff[i]}")
            discr += 1
        elif verbose:
            print(f"No discrepancy found at var {i} and instance {data_instance}:\n\t\tSIMICE p_val: {mdni_pval[i]}\n\t\tSequre p_val: {sequre_pval[i]}\n\t\tSIMICE coeff: {mdni_coeff[i]}\n\t\tSequre coeff: {sequre_coeff[i]}")
    
    return discr

def count_discrepancies(imputed_datasets, mice_imputed_data, labels):
    mice_stats = get_stat_sig(mice_imputed_data, labels)

    discr = 0
    for i, imputed_dataset in enumerate(imputed_datasets):
        sequre_stats = get_stat_sig(imputed_dataset, labels)
        discr += discrepancies(i, mice_stats, sequre_stats)
    
    return discr / len(imputed_datasets)

def evaluate_scenario(msg, scenario, impute_model, mi_factor, miss_col, miss_val, dump_data):
    (tampered_data, labels, non_tampered_data, miss_rows) = scenario

    mice_imputed_data = pymice(tampered_data)
    if dump_data:
        np.savetxt(f"../../data/mi/{msg}/py_mice_data.txt", mice_imputed_data, fmt='%10.2f')
    
    fit_model = LinearRegression()
    mi, imputed_datasets = MI(mi_factor, impute_model, fit_model).fit(
        np.nan_to_num(tampered_data, nan=miss_val), labels, miss_rows, miss_col, noise=True if msg == "scenario_1" else False)

    predicted_data = np.expand_dims(mi.model.predict(non_tampered_data), axis=1)
    stats = np.hstack((labels, predicted_data))
    bias = np.linalg.norm(predicted_data - labels)
    sd = np.mean(np.std(stats, axis=1))
    discrepancies_count = count_discrepancies(imputed_datasets, mice_imputed_data, labels)

    return mi.model.coef_, bias, sd, discrepancies_count

def benchmark_scenario(msg, runs_count, scenario, impute_model, mi_factor, miss_col, miss_val, dump_data):
    coefs = np.zeros((runs_count, 2))
    biases = np.zeros((runs_count,))
    sds = np.zeros((runs_count,))
    discrs = np.zeros((runs_count,))
    
    for i in tqdm(range(runs_count), "Evaluating scenario"):
        coef, bias, sd, discr = evaluate_scenario(
            msg, scenario, impute_model, mi_factor, miss_col, miss_val, dump_data)
        coefs[i] = coef
        biases[i] = bias
        sds[i] = sd
        discrs[i] = discr
    
    return coefs, biases, sds, discrs

def evaluate_real_setup(real_data, mi_factor, miss_val, dump_data):
    incomplete_data, complete_data, labels, miss_rows, col_types = real_data

    impute_models = [LogisticRegression() if t == "log" else LinearRegression() for t in col_types]
    fit_model = LinearRegression()
    
    mice_imputed_data = pymice(incomplete_data)
    if dump_data:
        np.savetxt(f"../../data/mi/real/py_mice_data.txt", mice_imputed_data, fmt='%10.2f')

    incompletes_w_miss_val = np.nan_to_num(incomplete_data, nan=miss_val)
    mi, imputed_datasets = BatchMI(mi_factor, impute_models, fit_model).fit(
        incompletes_w_miss_val, complete_data, labels, miss_rows)
    
    predicted_data = np.expand_dims(mi.model.predict(incompletes_w_miss_val), axis=1)
    stats = np.hstack((np.expand_dims(labels, axis=1), predicted_data))
    bias = np.linalg.norm(predicted_data - labels)
    sd = np.mean(np.std(stats, axis=1))
    discrepancies_count = count_discrepancies(imputed_datasets, mice_imputed_data, labels)

    return mi.model.coef_, bias, sd, discrepancies_count

def benchmark_real_setup(runs_count, real_data, mi_factor, miss_val, dump_data):
    biases = np.zeros((runs_count,))
    sds = np.zeros((runs_count,))
    discrs = np.zeros((runs_count,))
    
    for i in tqdm(range(runs_count), "Evaluating real setup"):
        _, bias, sd, discr = evaluate_real_setup(real_data, mi_factor, miss_val, dump_data)
        biases[i] = bias
        sds[i] = sd
        discrs[i] = discr
    
    return None, biases, sds, discrs

def print_stats(msg, coefs, biases, sds, discrepancies_count):
    print(f"{msg} | Python MI bias/rMSE w.r.t. ground truth | mean: {np.mean(biases)}, min: {np.min(biases)}, max: {np.max(biases)}")
    print(f"{msg} | Python MI SD w.r.t. ground truth | mean: {np.mean(sds)}, min: {np.min(sds)}, max: {np.max(sds)}")
    print(f"{msg} | Python MI # of discrepancies w.r.t. PyMICE per imputed dataset | mean: {np.mean(discrepancies_count)}, min: {np.min(discrepancies_count)}, max: {np.max(discrepancies_count)}")
    
    if msg.lower().startswith("scenario"):
        coefs_truth = np.ones_like(coefs[0])
        coefs_bias = np.linalg.norm(np.mean(coefs, axis=0) - coefs_truth)
        coefs_sd = np.mean(np.std(coefs, axis=0))
        coefs_rmse = np.mean(np.sum((coefs - coefs_truth) ** 2, axis=1))

        print(f"{msg} | Python MI coefs bias: {coefs_bias}")
        print(f"{msg} | Python MI coefs SD: {coefs_sd}")
        print(f"{msg} | Python MI coefs rMSE: {coefs_rmse}")

### Setup

In [45]:
mi_factor = 5
miss_val = 0.0
dump_data = True

scenarios_runs_count = 1000
scenarios_rows = 500
scenarios_miss_col = 0

real_setup_runs_count = 5
real_setup_outlier_threshold = 3000.0

### Data prep

In [46]:
# Scenario 1
sc1 = generate_scenario_data(scenarios_rows, "linear", scenarios_miss_col)

Generated data missing rate: 259/500


In [47]:
# Scenario 2
sc2 = generate_scenario_data(scenarios_rows, "logistic", scenarios_miss_col)

Generated data missing rate: 247/500


In [48]:
# Real data
real = parse_real_data(real_setup_outlier_threshold)

/tmp/ipykernel_947818/762480719.py:36: DtypeWarning: Columns (197,198) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/mi/real/gcasr.csv")
/tmp/ipykernel_947818/762480719.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_of_interest[binary_columns] = df_of_interest[binary_columns] * 2 - 1
/tmp/ipykernel_947818/762480719.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_of_interest[continuous_columns] = (df_of_interest[continuous_columns] -

In [49]:
if dump_data:
    export_data("scenario_1", *sc1, miss_val)
    export_data("scenario_2", *sc2, miss_val)

### Imputation via regression analysis

In [50]:
print_stats("Scenario 1", *benchmark_scenario("scenario_1", scenarios_runs_count, sc1, LinearRegression(), mi_factor, scenarios_miss_col, miss_val, dump_data))
print_stats("Scenario 2", *benchmark_scenario("scenario_2", scenarios_runs_count, sc2, LogisticRegression(), mi_factor, scenarios_miss_col, miss_val, dump_data))
print_stats("Real data", *benchmark_real_setup(real_setup_runs_count, real, mi_factor, miss_val, dump_data))

Evaluating scenario: 100%|██████████| 1000/1000 [00:11<00:00, 84.48it/s]


Scenario 1 | Python MI bias/rMSE w.r.t. ground truth | mean: 23.96946625969501, min: 23.73812967361939, max: 24.293834360697716
Scenario 1 | Python MI SD w.r.t. ground truth | mean: 0.43032933635506726, min: 0.42638406041769134, max: 0.4355428523971177
Scenario 1 | Python MI # of discrepancies w.r.t. PyMICE per imputed dataset | mean: 0.0, min: 0.0, max: 0.0
Scenario 1 | Python MI coefs bias: 0.28208839528855845
Scenario 1 | Python MI coefs SD: 0.013545505489680955
Scenario 1 | Python MI coefs rMSE: 0.0799655349609645


Evaluating scenario: 100%|██████████| 1000/1000 [00:13<00:00, 75.49it/s]


Scenario 2 | Python MI bias/rMSE w.r.t. ground truth | mean: 23.16425886402624, min: 23.164258864026237, max: 23.164258864026237
Scenario 2 | Python MI SD w.r.t. ground truth | mean: 0.4155631049040843, min: 0.4155631049040843, max: 0.4155631049040843
Scenario 2 | Python MI # of discrepancies w.r.t. PyMICE per imputed dataset | mean: 0.0, min: 0.0, max: 0.0
Scenario 2 | Python MI coefs bias: 0.2704117363717053
Scenario 2 | Python MI coefs SD: 3.4416913763379853e-15
Scenario 2 | Python MI coefs rMSE: 0.07312250716756032


Evaluating real setup:   0%|          | 0/5 [00:00<?, ?it/s]/home/hsmajlovic/miniconda3/envs/geometry/lib/python3.8/site-packages/sklearn/impute/_iterative.py:699: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
Evaluating real setup:  20%|██        | 1/5 [00:16<01:07, 16.80s/it]/home/hsmajlovic/miniconda3/envs/geometry/lib/python3.8/site-packages/sklearn/impute/_iterative.py:699: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
Evaluating real setup:  40%|████      | 2/5 [00:32<00:47, 15.92s/it]/home/hsmajlovic/miniconda3/envs/geometry/lib/python3.8/site-packages/sklearn/impute/_iterative.py:699: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
Evaluating real setup:  60%|██████    | 3/5 [00:47<00:31, 15.50s/it]/home/hsmajlovic/miniconda3/envs/geometry/lib/python3.8/site-packages/sklearn/impute/_iterative.py:699: ConvergenceWarning: [IterativeImputer]

Real data | Python MI bias/rMSE w.r.t. ground truth | mean: 87165.13495014918, min: 87162.10267917735, max: 87168.3415407975
Real data | Python MI SD w.r.t. ground truth | mean: 0.31210235141970877, min: 0.3113777044599365, max: 0.31290125772343536
Real data | Python MI # of discrepancies w.r.t. PyMICE per imputed dataset | mean: 0.96, min: 0.8, max: 1.0
